# PyProgressive — Progressive Visualization

This notebook demonstrates `pp.vis`, which binds PyProgressive's progressive
computation to interactive Plotly charts.

All visualization uses the **generator loop** pattern:

```python
for state in program.run(interval=0.5):
    fig, ax = pp.vis.subplots()
    ax.line(state.value(mean), label="Mean")
```

- `program.run(interval)` yields an `IterState` object every `interval` seconds
- `state.value(var)` returns the current progressive estimate for a variable
- `pp.vis.subplots(nrows, ncols)` returns a `(fig, axes)` pair (reused across ticks)
- Axes methods: `ax.line()`, `ax.scatter()`, `ax.bar()`, `ax.pie()`, `ax.heatmap()`,
  `ax.axhline()`, `ax.axvline()`


In [1]:
%pip install -e ..
%pip install plotly anywidget

Obtaining file:///C:/Users/shoon/Desktop/PyProgressive/PyProgressive
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for pyprogressive (pyproject.toml): started
  Building editable for pyprogressive (pyproject.toml): finished with status 'done'
  Created wheel for pyprogressive: filename=pyprogressive-0.1-0.editable-py3-none-any.whl size=2750 sha256=58d187d9e2339f3ec1d4b30e0363fdc9a3127fac33dcc2bcc70ebeef5e285dbf
  Stored in directory: C:\Users\shoon\AppData\Local\Temp\pip-ephem-wheel-cache-i2yg5m5m\wheels\45\8b\9e\eea62cd1f40


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 1. Line Chart — Basic Example

Compute mean and variance of a small array progressively.
The chart updates in real-time as each data point is processed.


In [2]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset() # clear arrays from previous cells

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["loan_amnt"],
)
df = df.dropna()

data = pp.array(df["loan_amnt"])

mean = accum(each(data)) / len(data)
var  = accum((each(data) - mean) ** 2) / len(data)

program = pp.compile(mean, var)

for state in program.run(interval=1):
    fig, ax = pp.vis.subplots()
    ax.set_title("Progressive Mean & Variance")
    ax.set_xlabel("Elapsed Time (s)")
    ax.set_ylabel("Value")
    ax.line(state.value(mean), label="Mean")
    ax.line(state.value(var),  label="Variance")


---
## 2. Line Chart — California Housing Dataset

Compute covariance and variance for multiple features progressively.
Each statistic is one line; watch them converge as more data is processed.


In [3]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayX2 = pp.array(X["HouseAge"].tolist())
arrayY  = pp.array(y.tolist())

mean1  = accum(each(arrayX1)) / len(arrayX1)
mean2  = accum(each(arrayX2)) / len(arrayX2)
meanY  = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
var2   = accum((each(arrayX2) - mean2) ** 2) / len(arrayX2)

covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)
covX2Y = accum((each(arrayX2) - mean2) * (each(arrayY) - meanY)) / len(arrayX2)

program = pp.compile(covX1Y, covX2Y, var1, var2)

for state in program.run(interval=0.3):
    pp.vis.line(state.value(covX1Y), label="Cov(MedInc, Price)",
                title="California Housing — Progressive Statistics",
                xlabel="Elapsed Time (s)")
    pp.vis.line(state.value(covX2Y), label="Cov(HouseAge, Price)")
    pp.vis.line(state.value(var1),   label="Var(MedInc)")
    pp.vis.line(state.value(var2),   label="Var(HouseAge)")


---
## 3. Scatter Chart — Convergence Trajectory

`ax.scatter(x, y)` adds one `(x, y)` point per tick.
Plotting two converging statistics against each other shows the
convergence trajectory — the path from early noisy estimates to the true value.


In [4]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayY  = pp.array(y.tolist())

mean1 = accum(each(arrayX1)) / len(arrayX1)
meanY = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)

program = pp.compile(covX1Y, var1)

for state in program.run(interval=0.3):
    pp.vis.scatter(state.value(covX1Y), state.value(var1),
                   title="Convergence Trajectory — Cov vs Var",
                   xlabel="Cov(MedInc, Price)", ylabel="Var(MedInc)")


---
## 4. subplots() — Single Subplot

`pp.vis.subplots()` returns `(fig, ax)`, just like `plt.subplots()`.
Call it inside the loop; the same `LiveAxes` object is returned each tick
so data accumulates automatically.


In [5]:
import pyprogressive as pp
from pyprogressive import each, accum

pp.reset()

data = pp.array([10, 20, 0, 21, 5, 42, 11, 14, 34, 13])

mean = accum(each(data)) / len(data)
var  = accum((each(data) - mean) ** 2) / len(data)

program = pp.compile(mean, var)

for state in program.run(interval=0.0001):
    pp.vis.line(state.value(mean), label="Mean",
                title="Progressive Mean & Variance",
                xlabel="Elapsed Time (s)", ylabel="Value")
    pp.vis.line(state.value(var),  label="Variance")


---
## 5. subplots(1, 2) — Line + Scatter Side by Side

`pp.vis.subplots(1, 2)` returns `(fig, [ax1, ax2])`.
Left panel: two statistics as lines. Right panel: convergence trajectory scatter.


In [6]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"])
arrayY  = pp.array(y)

mean1 = accum(each(arrayX1)) / len(arrayX1)
meanY = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)

program = pp.compile(covX1Y, var1)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2)
    ax1.set_title("Lines over Time")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.line(state.value(covX1Y), label="Cov(MedInc, Price)")
    ax1.line(state.value(var1),   label="Var(MedInc)")
    ax2.set_title("Convergence Trajectory")
    ax2.set_xlabel("Cov(MedInc, Price)")
    ax2.set_ylabel("Var(MedInc)")
    ax2.scatter(state.value(covX1Y), state.value(var1))


---
## 6. Bar Chart — GroupBy Variable

`ax.bar(values)` accepts a **dict** (GroupBy result) or a scalar.
Each group becomes a bar; the heights update as a snapshot on every tick.


In [7]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G

pp.reset()

data = pp.array([
    ('A', 2), ('B', 1), ('B', 4), ('C', 3), ('A', -3),
    ('A', 10), ('A', 8), ('B', 7), ('A', 10), ('A', 0),
])

group_sum  = group(each(data, 0), accum(each(G, 1)))
group_mean = group(each(data, 0), accum(each(G, 1)) / accum(1))

program = pp.compile(group_sum, group_mean)

for state in program.run(interval=0.05):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2)
    ax1.set_title("Group Sum")
    ax1.set_ylabel("Sum")
    ax1.bar(state.value(group_sum),  label="Sum")
    ax2.set_title("Group Mean")
    ax2.set_ylabel("Mean")
    ax2.bar(state.value(group_mean), label="Mean")


---
## 7. Bar Chart — Current Snapshot as Named Bars

Pass a **dict** to `ax.bar()` to display named bars in a single trace.
The chart always shows the latest estimate — a real-time snapshot.


In [8]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X['MedInc'])
arrayX2 = pp.array(X['HouseAge'])
arrayX3 = pp.array(X['AveRooms'])
arrayY  = pp.array(y)

mean1 = accum(each(arrayX1)) / len(arrayX1)
mean2 = accum(each(arrayX2)) / len(arrayX2)
mean3 = accum(each(arrayX3)) / len(arrayX3)
meanY = accum(each(arrayY))  / len(arrayY)

covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)
covX2Y = accum((each(arrayX2) - mean2) * (each(arrayY) - meanY)) / len(arrayX2)
covX3Y = accum((each(arrayX3) - mean3) * (each(arrayY) - meanY)) / len(arrayX3)

program = pp.compile(covX1Y, covX2Y, covX3Y)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2)
    # Left: current covariances as a grouped bar (dict with named keys)
    ax1.set_title('Covariances (current estimate)')
    ax1.bar({
        'Cov(MedInc,Price)':   state.value(covX1Y),
        'Cov(HouseAge,Price)': state.value(covX2Y),
        'Cov(AveRooms,Price)': state.value(covX3Y),
    })
    # Right: convergence over time as lines
    ax2.set_title('Covariances (convergence over time)')
    ax2.set_xlabel('Elapsed Time (s)')
    ax2.line(state.value(covX1Y), label='Cov(MedInc,Price)')
    ax2.line(state.value(covX2Y), label='Cov(HouseAge,Price)')
    ax2.line(state.value(covX3Y), label='Cov(AveRooms,Price)')


---
## 8. Loan Dataset — Covariance & Correlation

`pp.sqrt()` lets you build derived PyProgressive variables — ideal for **Pearson correlation**:

```python
corr = cov_xy / pp.sqrt(var_x * var_y)
```

This is a regular variable: pass it to `compile()` and plot with `ax.line(state.value(corr))`.

Left panel: covariances over time. Right panel: Pearson correlations over time.


In [9]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["loan_amnt", "installment", "annual_inc"],
    nrows=50000,
)
df = df.dropna()

X1 = pp.array(df["loan_amnt"])
X2 = pp.array(df["installment"])
X3 = pp.array(df["annual_inc"])

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1  = accum((each(X1) - mean1) ** 2) / len(X1)
var2  = accum((each(X2) - mean2) ** 2) / len(X2)
var3  = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, corr12, corr13)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2)
    ax1.set_title("Covariance — Loan Dataset")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.line(state.value(cov12), label="Cov(LoanAmt, Installment)")
    ax1.line(state.value(cov13), label="Cov(LoanAmt, AnnualInc)")
    ax2.set_title("Pearson Correlation — Loan Dataset")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.set_ylabel("Correlation")
    ax2.line(state.value(corr12), label="Corr(LoanAmt, Installment)")
    ax2.line(state.value(corr13), label="Corr(LoanAmt, AnnualInc)")


---
## 9. Yellow Taxi Trip Data — Covariance & Correlation

NYC taxi trips (100k rows).
- **Cov(Distance, Fare)** should be large and positive — longer rides cost more.
- **Corr(Distance, Fare)** converges toward ~0.9, confirming the strong linear link.
- **Corr(Distance, Tip)** is weaker, reflecting variable tipping behaviour.


In [10]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["trip_distance", "fare_amount", "tip_amount"],
)
df = df.dropna()
df = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]

X1 = pp.array(df["trip_distance"])
X2 = pp.array(df["fare_amount"])
X3 = pp.array(df["tip_amount"])

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1  = accum((each(X1) - mean1) ** 2) / len(X1)
var2  = accum((each(X2) - mean2) ** 2) / len(X2)
var3  = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, corr12, corr13)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2)
    ax1.set_title("Covariance — Yellow Taxi Trips")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.line(state.value(cov12), label="Cov(Distance, Fare)")
    ax1.line(state.value(cov13), label="Cov(Distance, Tip)")
    ax2.set_title("Pearson Correlation — Yellow Taxi Trips")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.set_ylabel("Correlation")
    ax2.line(state.value(corr12), label="Corr(Distance, Fare)")
    ax2.line(state.value(corr13), label="Corr(Distance, Tip)")


---
## 10. GroupBy Bar Chart — Mean Loan Amount by Grade

`ax.bar(state.value(group_mean))` renders a **GroupBy variable** result as a bar chart.
Each group shows the current progressive mean estimate.


In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["grade", "loan_amnt"],
    nrows=50000,
)
df = df.dropna()

data = pp.array(zip(df["grade"], df["loan_amnt"]))

group_mean = group(each(data, 0), accum(each(G, 1)) / accum(1))

program = pp.compile(group_mean)

for state in program.run(interval=1):
    pp.vis.bar(state.value(group_mean), label="Mean Loan Amount",
               title="Mean Loan Amount by Grade",
               ylabel="$ (USD)", figsize=(700, 450))


---
## 11. Mean & Variance over Time — Yellow Taxi Fares

Track how the sample mean and variance of fare amount converge
as more rows are processed.


In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["fare_amount"],
)
df = df.dropna()
df = df[df["fare_amount"] > 0]

X = pp.array(df["fare_amount"])

mean = accum(each(X)) / len(X)
var  = accum((each(X) - mean) ** 2) / len(X)

program = pp.compile(mean, var)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1000, 450))
    ax1.set_title("Mean Fare Amount")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.set_ylabel("Fare Amount (USD)")
    ax1.line(state.value(mean), label="Mean Fare")
    ax2.set_title("Variance of Fare Amount")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.set_ylabel("Variance")
    ax2.line(state.value(var), label="Variance")
    fig.suptitle("Yellow Taxi — Mean & Variance (100k rows)")


---
## 12. Heatmap — Progressive Correlation Matrix

`ax.heatmap(z, labels, zmin, zmax)` accepts a 2-D list of **current scalar values**
(e.g. `state.value(corr12)`) and numeric constants (e.g. `1` on the diagonal).
The matrix is a snapshot rendered fresh each tick.

Left: 4×4 correlation matrix as a heatmap.  
Right: off-diagonal entries as convergence lines.


In [3]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()

data = fetch_california_housing(as_frame=True)
X = data.data

X1 = pp.array(X["MedInc"].tolist())
X2 = pp.array(X["HouseAge"].tolist())
X3 = pp.array(X["AveRooms"].tolist())
X4 = pp.array(X["AveOccup"].tolist())

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)
mean4 = accum(each(X4)) / len(X4)

var1 = accum((each(X1) - mean1) ** 2) / len(X1)
var2 = accum((each(X2) - mean2) ** 2) / len(X2)
var3 = accum((each(X3) - mean3) ** 2) / len(X3)
var4 = accum((each(X4) - mean4) ** 2) / len(X4)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)
cov14 = accum((each(X1) - mean1) * (each(X4) - mean4)) / len(X1)
cov23 = accum((each(X2) - mean2) * (each(X3) - mean3)) / len(X2)
cov24 = accum((each(X2) - mean2) * (each(X4) - mean4)) / len(X2)
cov34 = accum((each(X3) - mean3) * (each(X4) - mean4)) / len(X3)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)
corr14 = cov14 / pp.sqrt(var1 * var4)
corr23 = cov23 / pp.sqrt(var2 * var3)
corr24 = cov24 / pp.sqrt(var2 * var4)
corr34 = cov34 / pp.sqrt(var3 * var4)

program = pp.compile(corr12, corr13, corr14, corr23, corr24, corr34)

LABELS = ["MedInc", "HouseAge", "AveRooms", "AveOccup"]

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1100, 480))
    r12 = state.value(corr12)
    r13 = state.value(corr13)
    r14 = state.value(corr14)
    r23 = state.value(corr23)
    r24 = state.value(corr24)
    r34 = state.value(corr34)
    ax1.heatmap(
        [[1,   r12, r13, r14],
         [r12, 1,   r23, r24],
         [r13, r23, 1,   r34],
         [r14, r24, r34, 1  ]],
        labels=LABELS, zmin=-1, zmax=1,
    )
    ax1.set_title("Correlation Matrix (Progressive)")
    ax2.line(r12, label="Corr(MedInc, HouseAge)")
    ax2.line(r13, label="Corr(MedInc, AveRooms)")
    ax2.line(r14, label="Corr(MedInc, AveOccup)")
    ax2.set_title("Off-diagonal Correlations over Time")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.set_ylabel("Pearson r")
    fig.suptitle("California Housing — 4×4 Correlation Matrix")


---
## 13. Pie / Donut Chart — Progressive GroupBy Proportions

`ax.pie(state.value(group_var))` renders a **GroupBy variable** result as a
pie or donut chart snapshot.  `hole=0.45` makes it a donut.

Three panels: pie (count), donut (sum), bar (mean).


In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum, group, G
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "loan.csv"),
    usecols=["grade", "loan_amnt"],
    nrows=50000,
)
df = df.dropna()

data = pp.array(zip(df["grade"], df["loan_amnt"]))

group_count = group(each(data, 0), accum(1))
group_mean  = group(each(data, 0), accum(each(G, 1)) / accum(1))
group_sum   = group(each(data, 0), accum(each(G, 1)))

program = pp.compile(group_count, group_mean, group_sum)

for state in program.run(interval=1.0):
    fig, (ax1, ax2, ax3) = pp.vis.subplots(1, 3, figsize=(1300, 450))
    ax1.pie(state.value(group_count))
    ax1.set_title("Loan Count by Grade")
    ax2.pie(state.value(group_sum), hole=0.45)
    ax2.set_title("Loan Sum by Grade")
    ax3.bar(state.value(group_mean))
    ax3.set_title("Mean Loan Amount by Grade")
    ax3.set_ylabel("$ (USD)")
    fig.suptitle("Loan Dataset — Grade Distribution & Mean Amount")


---
## 14. axhline / axvline — Reference Lines

`ax.axhline(y)` draws a horizontal line at a fixed *y* value across the full subplot.
`ax.axvline(x)` draws a vertical line at a fixed *x* value.

These are declared inside the loop body each tick, so they update in sync with the chart.
Useful for showing boundaries (e.g. ±1 for correlation, 0 baseline for covariance).


In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
import pandas as pd
import os

pp.reset()

DATA_DIR = os.path.join(os.path.dirname(pp.__file__), "data")
df = pd.read_csv(
    os.path.join(DATA_DIR, "yellow_tripdata_2016-01_100k.csv"),
    usecols=["trip_distance", "fare_amount", "tip_amount"],
)
df = df.dropna()
df = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]

X1 = pp.array(df["trip_distance"])
X2 = pp.array(df["fare_amount"])
X3 = pp.array(df["tip_amount"])

mean1 = accum(each(X1)) / len(X1)
mean2 = accum(each(X2)) / len(X2)
mean3 = accum(each(X3)) / len(X3)

var1 = accum((each(X1) - mean1) ** 2) / len(X1)
var2 = accum((each(X2) - mean2) ** 2) / len(X2)
var3 = accum((each(X3) - mean3) ** 2) / len(X3)

cov12 = accum((each(X1) - mean1) * (each(X2) - mean2)) / len(X1)
cov13 = accum((each(X1) - mean1) * (each(X3) - mean3)) / len(X1)

corr12 = cov12 / pp.sqrt(var1 * var2)
corr13 = cov13 / pp.sqrt(var1 * var3)

program = pp.compile(cov12, cov13, corr12, corr13)

for state in program.run(interval=0.3):
    fig, (ax1, ax2) = pp.vis.subplots(1, 2, figsize=(1100, 450))
    ax1.line(state.value(cov12), label="Cov(Distance, Fare)")
    ax1.line(state.value(cov13), label="Cov(Distance, Tip)")
    ax1.axhline(0, color="gray", linestyle="dash", linewidth=1)
    ax1.set_title("Covariance — Yellow Taxi Trips")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.set_ylabel("Covariance")
    ax2.line(state.value(corr12), label="Corr(Distance, Fare)")
    ax2.line(state.value(corr13), label="Corr(Distance, Tip)")
    ax2.axhline( 0,   color="gray",  linestyle="dash",     linewidth=1)
    ax2.axhline( 1,   color="green", linestyle="longdash", linewidth=1)
    ax2.axhline(-1,   color="green", linestyle="longdash", linewidth=1)
    ax2.axhline( 0.5, color="blue",  linestyle="dot",      linewidth=1)
    ax2.set_title("Pearson Correlation — Reference Lines")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.set_ylabel("Correlation")
    ax2.set_ylim(-1.1, 1.1)
    fig.suptitle("Yellow Taxi — axhline Demo")
